# Pipeline Test
This notebook runs the data ingestion and cleaning pipeline from `src.clean` to verify its execution.

In [1]:
import sys
import os
import pandas as pd

# Add the src directory to the python path so we can import from it
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

import clean

## Run Pipeline
The `process_data` function in `clean.py` loads raw data, validates it, cleans it, and outputs the merged dataset to `data/interim/processed_dataset.csv`.

In [2]:
# Execute the preprocessing pipeline
clean.process_data()

Loading datasets...
Validating raw data schemas...
Standardizing temporal fields...
Cleaning sensor data...
Cleaning weather data...
Aligning and Merging datasets...
Expanding vehicle_type_dist JSON...
Saving processed dataset to c:\Users\ojasd\projects\Flow-Cast\data\interim\processed_dataset.csv...
Data processing complete. Final shape: (176701, 29)


## Verification
Load the resulting dataset to verify it looks correct.

In [3]:
data_path = os.path.join('..', 'data', 'interim', 'processed_dataset.csv')

if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print(f"Processed dataset shape: {df.shape}")
    display(df.head())
else:
    print("Processed dataset was not found.")

Processed dataset shape: (176701, 29)


,road_id,road_name,latitude,longitude,weather_station_id,traffic_volume,vehicle_count,avg_speed,occupancy,congestion_level,...,visibility,public_holiday,holiday_name,event_flag,event_name,roadwork_flag,2W,Car,LCV,HCV
0,NL-001,Northline Ave @ 5th,28.643422,77.190411,WS-NORTH,97.0,97,53.0,7.7,Free-flow,...,10000.0,1,New Year's Day,0,NaN,0,0.39,0.35,0.12,0.14
1,NL-001,Northline Ave @ 5th,28.643422,77.190411,WS-NORTH,99.0,99,56.7,6.8,Unknown,...,10000.0,1,New Year's Day,0,NaN,0,0.44,0.34,0.11,0.12
2,NL-001,Northline Ave @ 5th,28.643422,77.190411,WS-NORTH,102.0,102,55.7,7.6,Free-flow,...,10000.0,1,New Year's Day,0,NaN,0,0.44,0.43,0.06,0.07
3,NL-001,Northline Ave @ 5th,28.643422,77.190411,WS-NORTH,113.0,113,53.5,7.6,Free-flow,...,10000.0,1,New Year's Day,0,NaN,0,0.43,0.36,0.09,0.13
4,NL-001,Northline Ave @ 5th,28.643422,77.190411,WS-NORTH,92.0,92,53.9,7.2,Free-flow,...,10000.0,1,New Year's Day,0,NaN,0,0.42,0.39,0.09,0.10


In [4]:
df.dtypes

road_id                   str
road_name                 str
latitude              float64
longitude             float64
weather_station_id        str
traffic_volume        float64
vehicle_count           int64
avg_speed             float64
occupancy             float64
congestion_level          str
travel_time           float64
accident_count          int64
signal_timing           int64
road_capacity           int64
timestamp                 str
station_id                str
weather_condition         str
temperature           float64
rainfall              float64
visibility            float64
public_holiday          int64
holiday_name              str
event_flag              int64
event_name                str
roadwork_flag           int64
2W                    float64
Car                   float64
LCV                   float64
HCV                   float64
dtype: object

In [5]:
df.isna().sum()

road_id                    0
road_name                  0
latitude                   0
longitude                  0
weather_station_id         0
traffic_volume             0
vehicle_count              0
avg_speed                  7
occupancy               4344
congestion_level           0
travel_time                0
accident_count             0
signal_timing              0
road_capacity              0
timestamp                  0
station_id             40974
weather_condition      40974
temperature            43134
rainfall               40974
visibility             42377
public_holiday             0
holiday_name          169690
event_flag                 0
event_name            169663
roadwork_flag              0
2W                         0
Car                        0
LCV                        0
HCV                        0
dtype: int64

## Handling Missing Values

In [6]:
df_clean=clean.handle_missing_values(df)

In [7]:
df_clean.isna().sum()

road_id               0
road_name             0
latitude              0
longitude             0
weather_station_id    0
traffic_volume        0
vehicle_count         0
avg_speed             0
occupancy             0
congestion_level      0
travel_time           0
accident_count        0
signal_timing         0
road_capacity         0
timestamp             0
weather_condition     0
temperature           0
rainfall              0
visibility            0
public_holiday        0
holiday_name          0
event_flag            0
event_name            0
roadwork_flag         0
2W                    0
Car                   0
LCV                   0
HCV                   0
dtype: int64

In [8]:
df_clean["weather_condition"].value_counts()

weather_condition
clear       101479
unknown      40974
cloudy       23553
rain          5360
overcast      4471
fog            864
Name: count, dtype: int64

In [9]:
df_clean.duplicated().sum()

np.int64(0)

In [10]:
df_clean=clean.normalised_timestamp(df_clean)


In [11]:
try:
    pd.to_datetime(df_clean['timestamp'], format='%d-%m-%Y', errors='raise')
    print("All dates match DD-MM-YYYY format.")
except ValueError as e:
    print(f"Format mismatch: {e}")

All dates match DD-MM-YYYY format.
